# 15 - Qwen Text + Video (SigLIP)

Modelo multimodal video + transcripcion con encoder de texto Qwen2.5 y encoder visual SigLIP.

In [1]:
import json
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_score

from transformers import AutoTokenizer, AutoModel, AutoProcessor
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
SEED = 42
GROUP_ID = 'BeingChillingWeWillWin'
MODEL_TAG = 'QwenText_SigLIPVideo_XGB_EN'
TEST_CASE = 'EXIST2026'
TRAIN_LANGS = ['en']

TEXT_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
VIDEO_MODEL_ID = 'google/siglip-base-patch16-224'
MAX_TEXT_LEN = 256
TEXT_BATCH = 8
NUM_FRAMES = 8

PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
WEIGHTS_DIR = PROJECT_ROOT / 'weights'
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_JSON = PROJECT_ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'training.json'
CLUSTER_JSON = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/training.json')
TRAIN_JSON_PATH = LOCAL_JSON if LOCAL_JSON.exists() else CLUSTER_JSON
CLUSTER_ROOT = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset')
VIDEO_ROOTS = [TRAIN_JSON_PATH.parent, TRAIN_JSON_PATH.parent.parent, CLUSTER_ROOT / 'training', CLUSTER_ROOT]

def resolve_video_abs_path(path_video: str) -> str:
    pv = Path(str(path_video))
    if pv.is_absolute() and pv.exists():
        return str(pv.resolve())
    cands = []
    for root in VIDEO_ROOTS:
        if not root.exists():
            continue
        cands.append((root / pv).resolve())
        cands.append((root / 'training' / pv).resolve())
    for c in cands:
        if c.exists():
            return str(c)
    return str(cands[0] if cands else pv.resolve())

def majority_task3_1(label_list):
    vals = [x for x in label_list if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    return vals[0] if c['YES'] == c['NO'] else c.most_common(1)[0][0]

with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)
rows = []
for sid, payload in raw.items():
    rec = {'sample_id': str(sid)}
    rec.update(payload)
    rows.append(rec)
df = pd.DataFrame(rows)
df['text'] = df['text'].fillna('').astype(str)
df['y'] = df['labels_task3_1'].apply(majority_task3_1).map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)
df['video_abs_path'] = df['path_video'].apply(resolve_video_abs_path)

train_mask = np.isin(df['lang'].astype(str).to_numpy(), TRAIN_LANGS) & (df['y'].to_numpy() >= 0)
split_upper = df['split'].astype(str).str.upper() if 'split' in df.columns else pd.Series([''] * len(df))
test_mask = split_upper.str.contains('TEST').to_numpy()
unlabeled_mask = df['y'].to_numpy() < 0
if test_mask.any():
    infer_mask = test_mask
elif unlabeled_mask.any():
    infer_mask = unlabeled_mask
elif (~train_mask).any():
    infer_mask = ~train_mask
else:
    infer_mask = np.ones(len(df), dtype=bool)

work = df.loc[train_mask | infer_mask, ['sample_id', 'video_abs_path', 'text', 'y']].reset_index(drop=True)
print('Rows for this run:', len(work))

Rows for this run: 2006


In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Video encoder
video_processor = AutoProcessor.from_pretrained(VIDEO_MODEL_ID)
video_model = AutoModel.from_pretrained(VIDEO_MODEL_ID).to(DEVICE)
video_model.eval()

def sample_frames(video_path: str, n: int = 8):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    idxs = np.linspace(0, max(total - 1, 0), num=n, dtype=int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok and frame is not None:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

def embed_video(path: str):
    if not Path(path).exists():
        return None
    frames = sample_frames(path, NUM_FRAMES)
    if not frames:
        return None
    inp = video_processor(images=frames, return_tensors='pt')
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    with torch.no_grad():
        feat = video_model.get_image_features(**inp)
        if hasattr(feat, 'pooler_output'):
            feat = feat.pooler_output
        elif isinstance(feat, (tuple, list)):
            feat = feat[0]
    return feat.mean(dim=0).detach().cpu().numpy().astype(np.float32)

video_embs = []
keep = []
for r in tqdm(work.itertuples(index=False), total=len(work), desc='Video embeddings'):
    e = embed_video(r.video_abs_path)
    video_embs.append(e)
    keep.append(e is not None)
work['video_emb'] = video_embs
work = work.loc[keep].reset_index(drop=True)

# Qwen text encoder
tok = AutoTokenizer.from_pretrained(TEXT_MODEL_ID, trust_remote_code=True)
dtype = torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32
text_model = AutoModel.from_pretrained(TEXT_MODEL_ID, torch_dtype=dtype, trust_remote_code=True, device_map='auto')
text_model.eval()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

text_emb_chunks = []
texts = work['text'].tolist()
for start in tqdm(range(0, len(texts), TEXT_BATCH), desc='Qwen embeddings'):
    batch = texts[start:start + TEXT_BATCH]
    enc = tok(batch, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
    enc = {k: v.to(text_model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = text_model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc['attention_mask'])
    text_emb_chunks.append(pooled.detach().cpu().numpy().astype(np.float32))

X_text = np.vstack(text_emb_chunks)
X_video = np.vstack(work['video_emb'].to_list()).astype(np.float32)
X = np.hstack([X_text, X_video]).astype(np.float32)
y = work['y'].to_numpy(dtype=np.int64)
ids = work['sample_id'].astype(str).to_numpy()

train_ids = set(df.loc[train_mask, 'sample_id'].astype(str).tolist())
infer_ids_set = set(df.loc[infer_mask, 'sample_id'].astype(str).tolist())
is_train = np.array([x in train_ids for x in ids], dtype=bool)
is_infer = np.array([x in infer_ids_set for x in ids], dtype=bool)
X_train, y_train = X[is_train], y[is_train]
X_infer, infer_ids = X[is_infer], ids[is_infer]

if HAS_XGB:
    clf = XGBClassifier(
        n_estimators=700, max_depth=6, learning_rate=0.03, subsample=0.9, colsample_bytree=0.8,
        objective='binary:logistic', eval_metric='logloss', random_state=SEED, n_jobs=-1
    )
else:
    clf = HistGradientBoostingClassifier(learning_rate=0.05, max_depth=10, random_state=SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
print('CV F1 macro mean:', float(scores.mean()))

clf.fit(X_train, y_train)
pred = clf.predict(X_infer).astype(int)
pred_labels = np.where(pred == 1, 'YES', 'NO')

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Video embeddings:   0%|          | 0/2006 [00:00<?, ?it/s]

[h264 @ 0x3b9dc880] Invalid NAL unit size (4049 > 274).
[h264 @ 0x3b9dc880] missing picture in access unit with size 278
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7207a0049fc0] stream 0, offset 0x100ebf: partial file
[h264 @ 0x3b79cbc0] Invalid NAL unit size (4049 > 274).
[h264 @ 0x3b79cbc0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7207a0049fc0] stream 1, offset 0x10136d: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7207a0049fc0] stream 0, offset 0x137203: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7207a0049fc0] stream 0, offset 0x137203: partial file
[h264 @ 0x3b9dc880] Invalid NAL unit size (4049 > 274).
[h264 @ 0x3b9dc880] missing picture in access unit with size 278
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7207a0049fc0] stream 0, offset 0x100ebf: partial file
[h264 @ 0x3b79cbc0] Invalid NAL unit size (4049 > 274).
[h264 @ 0x3b79cbc0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7207a0049fc0] stream 1, offset 0x10136d: partial file
[mov,mp4,m4a,3gp,3g2,mj2

[h264 @ 0x531a6800] Invalid NAL unit size (36106 > 59).
[h264 @ 0x531a6800] missing picture in access unit with size 63
[h264 @ 0x4f8baac0] Invalid NAL unit size (36106 > 59).
[h264 @ 0x4f8baac0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5327f7c0] stream 0, offset 0x617e9: partial file
[h264 @ 0x531a6800] Invalid NAL unit size (36106 > 59).
[h264 @ 0x531a6800] missing picture in access unit with size 63
[h264 @ 0x4f8baac0] Invalid NAL unit size (36106 > 59).
[h264 @ 0x4f8baac0] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5327f7c0] stream 0, offset 0x617e9: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5327f7c0] stream 0, offset 0x1785c0: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5327f7c0] stream 0, offset 0xebaf6: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5327f7c0] stream 0, offset 0xebaf6: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5327f7c0] stream 0, offset 0xebaf6: partial file
[h264 @ 0x531a6800] Invalid NAL unit size (36106 > 59).

...

[swscaler @ 0x5329e580] Unsupported input (Operation not supported): fmt:yuv420p csp:unknown prim:unknown trc:log316 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x5329e580] Unsupported input (Operation not supported): fmt:yuv420p csp:unknown prim:unknown trc:log316 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x5329e580] Unsupported input (Operation not supported): fmt:yuv420p csp:unknown prim:unknown trc:log316 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x5329e580] Unsupported input (Operation not supported): fmt:yuv420p csp:unknown prim:unknown trc:log316 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x5329e580] Unsupported input (Operation not supported): fmt:yuv420p csp:unknown prim:unknown trc:log316 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x5329e580] Unsupported input (Operation not supported): fmt:yuv420p csp:unknown prim:unknown trc:log316 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x5329e580

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen embeddings:   0%|          | 0/251 [00:00<?, ?it/s]

CV F1 macro mean: 0.7103853283581398


In [4]:
import joblib

model_path = WEIGHTS_DIR / f'{GROUP_ID}_{MODEL_TAG}.joblib'
joblib.dump(clf, model_path)

rows = []
for sid, lab in zip(infer_ids, pred_labels):
    rows.append({'test_case': TEST_CASE, 'id': str(sid), 'value': str(lab)})

json_path = DELIVERABLES_DIR / f'{GROUP_ID}_{MODEL_TAG}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False)

print('Saved model:', model_path)
print('Saved json:', json_path)
print('Rows:', len(rows))

Saved model: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/weights/BeingChillingWeWillWin_QwenText_SigLIPVideo_XGB_EN.joblib
Saved json: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/entregables/BeingChillingWeWillWin_QwenText_SigLIPVideo_XGB_EN.json
Rows: 1212
